In [3]:
import sys
import pandas as pd
from sklearn.model_selection import train_test_split

sys.path.append('..')

from src.preprocessing import build_preprocessing_pipeline

data = pd.read_csv('../data/digital_marketing_campaign_dataset.csv')

X = data.drop(columns=['Conversion', 'CustomerID'], errors='ignore')
y = data['Conversion']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

pipeline = build_preprocessing_pipeline()
X_train_processed = pipeline.fit_transform(X_train)

print(X_train_processed.shape)

(6400, 17)


In [4]:
feature_names = pipeline.named_steps['encoding_and_scaling'].get_feature_names_out()

X_train_processed_df = pd.DataFrame(
    X_train_processed, 
    columns=feature_names, 
    index=X_train.index
)

X_train_processed_df.head()

,num__Income,num__AdSpend,num__LoyaltyPoints,num__AdSpend_per_Visit,num__Engagement_Score,num__Loyalty_per_Income,cat__Gender_Female,cat__Gender_Male,cat__CampaignChannel_Email,cat__CampaignChannel_PPC,cat__CampaignChannel_Referral,cat__CampaignChannel_SEO,cat__CampaignChannel_Social Media,cat__CampaignType_Awareness,cat__CampaignType_Consideration,cat__CampaignType_Conversion,cat__CampaignType_Retention
2787,1.255559,0.900155,-0.857479,-0.119669,0.301486,-0.816048,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
7093,-0.542982,0.216438,0.813990,-0.119670,2.205419,0.491969,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
6379,0.936737,-1.655482,1.705579,-0.119672,-0.113306,0.057714,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3865,0.169332,0.658068,-0.369443,-0.119669,2.003322,-0.485093,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
1167,-0.572632,-1.036648,-0.209558,-0.119669,-0.886202,-0.122215,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0


In [5]:
X_train_processed_df.columns.tolist()

['num__Income',
 'num__AdSpend',
 'num__LoyaltyPoints',
 'num__AdSpend_per_Visit',
 'num__Engagement_Score',
 'num__Loyalty_per_Income',
 'cat__Gender_Female',
 'cat__Gender_Male',
 'cat__CampaignChannel_Email',
 'cat__CampaignChannel_PPC',
 'cat__CampaignChannel_Referral',
 'cat__CampaignChannel_SEO',
 'cat__CampaignChannel_Social Media',
 'cat__CampaignType_Awareness',
 'cat__CampaignType_Consideration',
 'cat__CampaignType_Conversion',
 'cat__CampaignType_Retention']

In [6]:
print("Missing values count:", X_train_processed_df.isna().sum().sum())

Missing values count: 0


In [7]:
import numpy as np 
X_train_processed_df.describe(include=[np.number]).T[['mean', 'std', 'min', 'max']]

,mean,std,min,max
num__Income,3.108624e-17,1.000078,-1.722207,1.740158
num__AdSpend,9.214851e-17,1.000078,-1.730690,1.758719
num__LoyaltyPoints,8.992806e-17,1.000078,-1.737897,1.752358
num__AdSpend_per_Visit,-1.332268e-17,1.000078,-0.119672,12.625200
num__Engagement_Score,1.088019e-16,1.000078,-1.279915,3.178537
num__Loyalty_per_Income,-5.551115e-17,1.000078,-1.080678,5.503384
cat__Gender_Female,6.050000e-01,0.488889,0.000000,1.000000
cat__Gender_Male,3.950000e-01,0.488889,0.000000,1.000000
cat__CampaignChannel_Email,1.957812e-01,0.396832,0.000000,1.000000
cat__CampaignChannel_PPC,2.071875e-01,0.405323,0.000000,1.000000


**Exploratory & Experimentation Phase**

Model Training & Baseline Evaluation

In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

log_reg = LogisticRegression(random_state=42, max_iter=1000)
log_reg.fit(X_train_processed, y_train)

y_train_pred = log_reg.predict(X_train_processed)
y_train_prob = log_reg.predict_proba(X_train_processed)[:, 1]

X_test_processed = pipeline.transform(X_test)

y_test_pred = log_reg.predict(X_test_processed)
y_test_prob = log_reg.predict_proba(X_test_processed)[:, 1]

print("=== Logistic Regression Train Performance ===")
print(classification_report(y_train, y_train_pred))
print(f"Train ROC-AUC Score: {roc_auc_score(y_train, y_train_prob):.4f}")

print("\n" + "="*40 + "\n")


print(" Logistic Regression Test Performance ...")
print(classification_report(y_test, y_test_pred))
print(f"Test ROC-AUC Score: {roc_auc_score(y_test, y_test_prob):.4f}")


=== Logistic Regression Train Performance ===
              precision    recall  f1-score   support

           0       0.00      0.00      0.00       790
           1       0.88      1.00      0.93      5610

    accuracy                           0.88      6400
   macro avg       0.44      0.50      0.47      6400
weighted avg       0.77      0.88      0.82      6400

Train ROC-AUC Score: 0.7107


 Logistic Regression Test Performance ...
              precision    recall  f1-score   support

           0       0.00      0.00      0.00       198
           1       0.88      1.00      0.93      1402

    accuracy                           0.88      1600
   macro avg       0.44      0.50      0.47      1600
weighted avg       0.77      0.88      0.82      1600

Test ROC-AUC Score: 0.6936


c:\Users\compu zone\miniconda3\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\compu zone\miniconda3\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\compu zone\miniconda3\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\c

**Balance for classes**

In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score

log_reg_balanced = LogisticRegression(
    class_weight='balanced', 
    random_state=42, 
    max_iter=1000
)

log_reg_balanced.fit(X_train_processed, y_train)

y_test_pred_bal = log_reg_balanced.predict(X_test_processed)
y_test_prob_bal = log_reg_balanced.predict_proba(X_test_processed)[:, 1]

print("=== Balanced Logistic Regression Test Performance ===")
print(classification_report(y_test, y_test_pred_bal))
print(f"Test ROC-AUC Score: {roc_auc_score(y_test, y_test_prob_bal):.4f}")

=== Balanced Logistic Regression Test Performance ===
              precision    recall  f1-score   support

           0       0.20      0.66      0.31       198
           1       0.93      0.64      0.75      1402

    accuracy                           0.64      1600
   macro avg       0.57      0.65      0.53      1600
weighted avg       0.84      0.64      0.70      1600

Test ROC-AUC Score: 0.6950


**Here We Use Random Forest Algorithm for best resulat ..**

In [11]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators= 100,
    random_state= 42,
    class_weight= 'balanced',
    n_jobs=-1
 )

rf_model.fit(X_train_processed, y_train)

y_train_pred_rf = rf_model.predict(X_train_processed)
y_train_prob_rf = rf_model.predict_proba(X_train_processed)[:, 1]

y_test_pred_rf = rf_model.predict(X_test_processed)
y_test_prob_rf = rf_model.predict_proba(X_test_processed)[:, 1]

print("=== Random Forest Train Performance ===")
print(classification_report(y_train, y_train_pred_rf))
print(f"Train ROC-AUC Score: {roc_auc_score(y_train, y_train_prob_rf):.4f}")

print("\n" + "="*40 + "\n")

print("=== Random Forest Test Performance ===")
print(classification_report(y_test, y_test_pred_rf))
print(f"Test ROC-AUC Score: {roc_auc_score(y_test, y_test_prob_rf):.4f}")

=== Random Forest Train Performance ===
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       790
           1       1.00      1.00      1.00      5610

    accuracy                           1.00      6400
   macro avg       1.00      1.00      1.00      6400
weighted avg       1.00      1.00      1.00      6400

Train ROC-AUC Score: 1.0000


=== Random Forest Test Performance ===
              precision    recall  f1-score   support

           0       0.41      0.24      0.31       198
           1       0.90      0.95      0.92      1402

    accuracy                           0.86      1600
   macro avg       0.66      0.60      0.62      1600
weighted avg       0.84      0.86      0.85      1600

Test ROC-AUC Score: 0.6908


**Hyperparameter Tuning / Regularization:**

In [12]:

rf_reg = RandomForestClassifier(
    n_estimators=100,
    max_depth=8,             
    min_samples_leaf=10,       
    min_samples_split=20,        
    class_weight='balanced',       
    random_state=42,
    n_jobs=-1
)

rf_reg.fit(X_train_processed, y_train)

y_train_pred_reg = rf_reg.predict(X_train_processed)
y_train_prob_reg = rf_reg.predict_proba(X_train_processed)[:, 1]

y_test_pred_reg = rf_reg.predict(X_test_processed)
y_test_prob_reg = rf_reg.predict_proba(X_test_processed)[:, 1]

print("=== Regularized Random Forest Train Performance ===")
print(classification_report(y_train, y_train_pred_reg))
print(f"Train ROC-AUC Score: {roc_auc_score(y_train, y_train_prob_reg):.4f}")

print("\n" + "="*40 + "\n")

print("=== Regularized Random Forest Test Performance ===")
print(classification_report(y_test, y_test_pred_reg))
print(f"Test ROC-AUC Score: {roc_auc_score(y_test, y_test_prob_reg):.4f}")

=== Regularized Random Forest Train Performance ===
              precision    recall  f1-score   support

           0       0.37      0.76      0.50       790
           1       0.96      0.82      0.88      5610

    accuracy                           0.81      6400
   macro avg       0.67      0.79      0.69      6400
weighted avg       0.89      0.81      0.84      6400

Train ROC-AUC Score: 0.8819


=== Regularized Random Forest Test Performance ===
              precision    recall  f1-score   support

           0       0.27      0.52      0.35       198
           1       0.92      0.80      0.86      1402

    accuracy                           0.76      1600
   macro avg       0.59      0.66      0.60      1600
weighted avg       0.84      0.76      0.79      1600

Test ROC-AUC Score: 0.7123


**Keep up the work so we can produce the best product.**

In [17]:
%pip install xgboost

Note: you may need to restart the kernel to use updated packages.


In [15]:
from xgboost import XGBClassifier

ratio = (y_train == 0).sum() / (y_train == 1).sum() # For fix Imbalance.
xgb_model = XGBClassifier(
    n_estimators=100,
    learning_rate=0.05,
    max_depth=4,
    scale_pos_weight=ratio,
    random_state=42,
    eval_metric='logloss'
 )

xgb_model.fit(X_train_processed, y_train)

y_test_pred_xgb = xgb_model.predict(X_test_processed)
y_test_prob_xgb = xgb_model.predict_proba(X_test_processed)[:, 1]

y_train_pred_xgb = xgb_model.predict(X_train_processed)
y_train_prob_xgb = xgb_model.predict_proba(X_train_processed)[:, 1]

print("=== XGBoost Test Performance ===")
print(classification_report(y_test, y_test_pred_xgb))
print(f"Test ROC-AUC Score: {roc_auc_score(y_test, y_test_prob_xgb):.4f}")

=== XGBoost Test Performance ===
              precision    recall  f1-score   support

           0       0.26      0.56      0.35       198
           1       0.92      0.77      0.84      1402

    accuracy                           0.75      1600
   macro avg       0.59      0.66      0.60      1600
weighted avg       0.84      0.75      0.78      1600

Test ROC-AUC Score: 0.7141


**Final Benchmark**

In [16]:
results = {
    "Model": ["Logistic Regression (Balanced)", "Regularized Random Forest", "XGBoost Classifier"],
    "Test ROC-AUC": [
        roc_auc_score(y_test, y_test_prob_bal),
        roc_auc_score(y_test, y_test_prob_reg),
        roc_auc_score(y_test, y_test_prob_xgb)
    ]
}

comparison_df = pd.DataFrame(results)
print("=== FINAL PROJECT BENCHMARK ===")
print(comparison_df.round(4))

=== FINAL PROJECT BENCHMARK ===
                            Model  Test ROC-AUC
0  Logistic Regression (Balanced)        0.6950
1       Regularized Random Forest        0.7123
2              XGBoost Classifier        0.7141


**"From a simple baseline to robust ensemble modeling: This pipeline reflects a deep, iterative process of understanding data, tackling class imbalance, and building a production-ready Machine Learning solution."**